# Working With Views
Demonstrates creating, querying, and dropping **persistent** and **temporary** views in Databricks using Spark SQL.

| View Type | Scope | Stored in Metastore |
|-----------|-------|---------------------|
| Persistent | Survives session restarts | Yes |
| Temporary | Current session only | No |
| Global Temporary | All sessions in the app | No (`global_temp` db) |

In [1]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark

spark.sql("SELECT current_catalog(), current_schema()").show(truncate=False)

+--------------------------------------+----------------+
|current_catalog()                     |current_schema()|
+--------------------------------------+----------------+
|dbw_dataengineering_sandbox_euwest_001|default         |
+--------------------------------------+----------------+



## Persistent View
A persistent view is stored in the metastore and survives session restarts.
It behaves like a virtual table defined by a query — no data is copied.
Similar to the SQL Server view although there is present MATERIALIZED options, and we could do indexing on top of that. 

In [2]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark

spark.sql("""
    CREATE OR REPLACE VIEW dbw_dataengineering_sandbox_euwest_001.default.vw_top10_countries_by_population AS
    SELECT *
    FROM dbw_dataengineering_sandbox_euwest_001.default.country_population
    ORDER BY POPULATION DESC
    LIMIT 10
""")

print("Persistent view created: vw_top10_countries_by_population")

Persistent view created: vw_top10_countries_by_population


In [3]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark

spark.sql("""
    SELECT *
    FROM dbw_dataengineering_sandbox_euwest_001.default.vw_top10_countries_by_population
""").show(truncate=False)

+----------+------------------------+-----------+----+----------+----------------+----------+-----------+---------+-------------+
|COUNTRY_ID|NAME                    |NATIONALITY|CODE|ISO_ALPHA3|CAPITAL         |POPULATION|AREA_KM2   |REGION_ID|SUB_REGION_ID|
+----------+------------------------+-----------+----+----------+----------------+----------+-----------+---------+-------------+
|45        |China                   |Chinese    |CHN |CN        |Beijing         |1433783686|9706961.0  |30       |60           |
|103       |India                   |Indian     |IND |IN        |New Delhi       |1366417754|3287590.0  |30       |30           |
|236       |United States of America|American   |USA |US        |Washington, D.C.|329064917 |9372610.0  |10       |80           |
|104       |Indonesia               |Indonesian |IDN |ID        |Jakarta         |270625568 |1904569.0  |30       |100          |
|167       |Pakistan                |Pakistani  |PAK |PK        |Islamabad       |21656531

## Temporary View
A temporary view is **session-scoped** — it exists only for the duration of the current Spark session
and is not stored in the metastore. No cleanup is required; it is dropped automatically when the session ends.

In [4]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark

spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW tmp_top10_countries AS
    SELECT *
    FROM dbw_dataengineering_sandbox_euwest_001.default.country_population
    ORDER BY POPULATION DESC
    LIMIT 10
""")

print("Temporary view created: tmp_top10_countries")

spark.sql("SELECT * FROM tmp_top10_countries").show(truncate=False)

Temporary view created: tmp_top10_countries
+----------+------------------------+-----------+----+----------+----------------+----------+-----------+---------+-------------+
|COUNTRY_ID|NAME                    |NATIONALITY|CODE|ISO_ALPHA3|CAPITAL         |POPULATION|AREA_KM2   |REGION_ID|SUB_REGION_ID|
+----------+------------------------+-----------+----+----------+----------------+----------+-----------+---------+-------------+
|45        |China                   |Chinese    |CHN |CN        |Beijing         |1433783686|9706961.0  |30       |60           |
|103       |India                   |Indian     |IND |IN        |New Delhi       |1366417754|3287590.0  |30       |30           |
|236       |United States of America|American   |USA |US        |Washington, D.C.|329064917 |9372610.0  |10       |80           |
|104       |Indonesia               |Indonesian |IDN |ID        |Jakarta         |270625568 |1904569.0  |30       |100          |
|167       |Pakistan                |Pakistani

## Showing Views
Use `SHOW VIEWS` to list all views in the current schema. Temporary views appear with `isTemporary = true`.

In [5]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark

spark.sql("""
    SHOW VIEWS IN dbw_dataengineering_sandbox_euwest_001.default
""").show(truncate=False)

+---------+--------------------------------+-----------+--------------+--------+
|namespace|viewName                        |isTemporary|isMaterialized|isMetric|
+---------+--------------------------------+-----------+--------------+--------+
|default  |vw_top10_countries_by_population|false      |false         |false   |
|         |tmp_top10_countries             |true       |false         |false   |
+---------+--------------------------------+-----------+--------------+--------+



## Drop the Persistent View
Persistent views must be explicitly dropped with `DROP VIEW`. `IF EXISTS` prevents an error if the view has already been removed.

In [6]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark

spark.sql("""
    DROP VIEW IF EXISTS dbw_dataengineering_sandbox_euwest_001.default.vw_top10_countries_by_population
""")

print("Persistent view dropped.")

Persistent view dropped.


In [7]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark

# After dropping the persistent view, only the temporary view should remain
spark.sql("""
    SHOW VIEWS IN dbw_dataengineering_sandbox_euwest_001.default
""").show(truncate=False)

+---------+-------------------+-----------+--------------+--------+
|namespace|viewName           |isTemporary|isMaterialized|isMetric|
+---------+-------------------+-----------+--------------+--------+
|         |tmp_top10_countries|true       |false         |false   |
+---------+-------------------+-----------+--------------+--------+

